# Kaggle cell worker — R1a host variant: SDXL + IP-Adapter, one cell at a time

Same locked recipe as `colab_sdxl_ipadapter.ipynb` (AP `docs/RIG_PLAN.md`: SDXL fp16 + IP-Adapter-plus on a T4-class 16 GB GPU; reach for ControlNet only if per-cell identity drifts beyond what the reference fixes), hosted on Kaggle's free T4. **This is a host variant of the existing lane, not a second generator**, and it stays out of the lane of record until Aden and the AP owner approve it. Bots never run it; the owner presses the buttons, exactly like Colab.


## 0. One-time setup (owner, in the Kaggle UI)

1. Phone-verify the Kaggle account; in notebook **Settings** enable **T4 x2** and **Internet**.
2. Download `leila_ipadapter_ref.png` from Drive (`MyDrive/HomeBot/leila_ipadapter_ref.png`) once and add it as a private notebook **Input**; set `REF_PATH` below to the path Kaggle shows for it.
3. No Google credentials enter this notebook: the reference is an attached input, and the output is downloaded by hand.


In [ ]:
# -- EDIT BEFORE RUNNING ------------------------------------------------------
# The locked Cell-1 figure prompt, from the AP owner. It MUST demand a solid
# #FF00FF magenta ground filling the frame behind ONE figure, flat cel style
# with outline, no props, no text. The attached reference carries identity.
PROMPT = 'TODO paste the locked Cell-1 prompt here'
NEGATIVE_PROMPT = ('blurry, distorted, low quality, artifacts, bad anatomy, deformed, watermark, '
                   'text, multiple figures, props, background scenery, gradient background')
REF_PATH = '/kaggle/input/leila_ipadapter_ref/leila_ipadapter_ref.png'
SEED = 42
WIDTH, HEIGHT = 1024, 1024   # AP compose_from_cells() rescales; a square cell is safe
NUM_STEPS, GUIDANCE, IP_SCALE = 30, 7.5, 0.7


In [ ]:
from pathlib import Path
import torch

assert 'TODO' not in PROMPT, 'PROMPT still has the TODO placeholder - paste the locked prompt first.'
assert ('FF00FF' in PROMPT.upper()) or ('magenta' in PROMPT.lower()), \
    'PROMPT must demand the solid #FF00FF ground - AP detection failed exactly there before.'
assert Path(REF_PATH).is_file(), (f'Reference not found: {REF_PATH}. Attach the ref as a notebook '
    'Input and fix REF_PATH. This worker REFUSES to generate without the reference - unlike the '
    'Colab worker, which only warns and continues, silently losing identity.')
assert torch.cuda.is_available(), 'Enable the T4 x2 accelerator in Settings first - CPU would waste the session.'
print(f'guards ok - ref={Path(REF_PATH).name} size={WIDTH}x{HEIGHT} seed={SEED}')


In [ ]:
%pip install -q -U diffusers transformers accelerate safetensors


In [ ]:
from diffusers import AutoPipelineForText2Image, DDIMScheduler
from PIL import Image

pipeline = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16, variant='fp16', use_safetensors=True)
pipeline.scheduler = DDIMScheduler.from_config(pipeline.scheduler.config)
pipeline.enable_attention_slicing()
pipeline.to('cuda')  # one T4 = the same 16 GB envelope as the Colab lane
pipeline.load_ip_adapter('h94/IP-Adapter', subfolder='sdxl_models',
                         weight_name='ip-adapter-plus_sdxl_vit-h.safetensors')
pipeline.set_ip_adapter_scale(IP_SCALE)
print('SDXL + IP-Adapter ready on GPU')


In [ ]:
import json
from datetime import datetime, timezone
import numpy as np

ref = Image.open(REF_PATH).convert('RGB')
gen = torch.Generator('cuda').manual_seed(SEED)
image = pipeline(prompt=PROMPT, negative_prompt=NEGATIVE_PROMPT, width=WIDTH, height=HEIGHT,
                num_inference_steps=NUM_STEPS, guidance_scale=GUIDANCE,
                ip_adapter_image=ref, generator=gen).images[0]

now = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out = Path('/kaggle/working') / f'cell_seed{SEED}_{WIDTH}x{HEIGHT}_{now}.png'
image.save(out)

# In-session ground check, mirroring AP's documented failure mode ('the generator
# did not draw a magenta border at all'): the frame border should be near #FF00FF.
a = np.asarray(image.convert('RGB')).astype(int)
border = np.concatenate([a[0], a[-1], a[:, 0], a[:, -1]])
near = (np.abs(border - np.array([255, 0, 255])).max(axis=1) <= 40)
frac = float(near.mean())
print(f'border magenta coverage: {frac:.1%}')
if frac < 0.9:
    print('WARN: the border is not solid #FF00FF - do NOT hand this cell to compose; fix the prompt.')

meta = {'prompt': PROMPT, 'negativePrompt': NEGATIVE_PROMPT, 'refPath': REF_PATH,
        'seed': SEED, 'width': WIDTH, 'height': HEIGHT, 'steps': NUM_STEPS,
        'guidance': GUIDANCE, 'ipAdapterScale': IP_SCALE,
        'model': 'stabilityai/stable-diffusion-xl-base-1.0',
        'ipAdapter': 'ip-adapter-plus_sdxl_vit-h.safetensors',
        'borderMagenta': round(frac, 4), 'generatedAt': now}
Path(str(out) + '.json').write_text(json.dumps(meta, indent=2))
print(f'saved {out.name} (+ .json metadata) in /kaggle/working - download both before the session ends.')


## 3. After the run (owner)

1. Download the PNG **and** its `.json` from the notebook output panel.
2. Put them in the Drive cell folder (or straight into AP's cell input), then run AP's compose + residual/speck verification.
3. One cell per session phase; Aden accepts or rejects. Only if identity drifts cell-to-cell beyond what the reference fixes does the locked recipe say to reach for ControlNet.
